# Can more candidates produce a better selected answer

Day10 · Chapter7 · actual bounded CPU candidate collection. A sampled correct answer is useful only if a deployable selector can identify it. This notebook compares selectors on identical pools, charges unsuccessful attempts, and makes the oracle boundary visible. English task descriptions are not parsed by the symbolic decoder.

[Chapter](../../book/chapters/07-evaluation-is-a-contract.md) · [Frozen protocol](../../experiments/specs/2026-10-04-inference-selection.md) · [Measured report](../../experiments/reports/2026-10-04-inference-selection.md)

Predict before each adjacent reference. Saved previews show historical observations; fresh sampling uses explicitly chosen coordinates, not a hidden best seed.

In [ ]:
from pathlib import Path
import sys, json
root=Path.cwd().resolve()
while not (root/"src"/"dongxi_llms").is_dir():
    if root.parent==root: raise RuntimeError("Open inside Dongxi_LLMs")
    root=root.parent
sys.path.insert(0,str(root/"src"))
import torch, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from dongxi_llms.inference_selection_lab import (
    fit_policy, generate_candidate, evaluate_pool, select, selection_view,
    budget_prefix, state_hash, SELECTORS, N_VALUES, TOKEN_BUDGETS)
torch.set_num_threads(1)
items=json.loads((root/"fixtures/inference-selection/items.json").read_text())
measured=json.loads((root/"experiments/reports/2026-10-04-inference-selection/results.json").read_text())
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,
 "figure.facecolor":"white","axes.facecolor":"white","axes.spines.top":False,"axes.spines.right":False})
colors={"first":"#777777","majority":"#356a9b","mean_logp":"#b97139","oracle":"#3d8268"}
def summary(policy,slice_name,kind,budget,method):
    return next(r for r in policy["summary"] if r["slice"]==slice_name and r["budget_kind"]==kind and r["budget"]==budget and r["method"]==method)
print("Three predeclared seeds:",measured["seeds"],"ordinary candidate errors:",len(measured["failures"]))


### Prediction

## Exercise1 Where may the gold answer enter

Trace an item through generation, parsing, selection and evaluation. Which box may compare the candidate with the reference? Could a likelihood selector be highly confident yet choose the wrong answer?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fig,ax=plt.subplots(figsize=(11,3.4))
ax.set(xlim=(0,12),ylim=(0,3.6));ax.axis("off")
boxes=[(.1,1.8,2.2,"Symbolic prompt\n4 IDs, no gold"),(3,1.8,2.3,"Tiny causal decoder\n8 seeded attempts"),(6,1.8,2.4,"Gold-blind selector\nfirst / vote / log p"),(9.2,1.8,2.6,"Selected candidate\none answer or abstain"),
       (6,.05,2.4,"Independent grader\nreference allowed"),(9.2,.05,2.6,"Any-correct oracle\ndiagnostic only")]
for x,y,w,label in boxes:
    patch=FancyBboxPatch((x,y),w,1.15,boxstyle="round,pad=.06",facecolor="#f2f5f7",edgecolor="#788995")
    ax.add_patch(patch);ax.text(x+w/2,y+.575,label,ha="center",va="center")
for x1,x2 in ((2.3,3),(5.3,6),(8.4,9.2)):
    ax.annotate("",(x2,2.375),(x1,2.375),arrowprops={"arrowstyle":"->","color":"#555555"})
ax.annotate("",(7.2,1.2),(4.15,1.8),arrowprops={"arrowstyle":"->","color":"#888888","linestyle":"--"})
ax.annotate("",(9.2,.625),(8.4,.625),arrowprops={"arrowstyle":"->","color":"#888888","linestyle":"--"})
ax.set_title("Architecture and evidence boundary schematic, not measured activations")
fig.tight_layout();plt.show()
probe=selection_view(measured["runs"][0]["policies"][0]["pools"][0]["graded_candidates"][0])
assert not {"reference","problem","correct","task_success","status"} & probe.keys()


### Reference explanation

Reference: generation and selection never receive a reference through their declared views. The independent evaluator labels every candidate after the decision. Any-correct availability is useful diagnosis but is not a deployable selector.

![Saved architecture and gold isolation](../figures/chapter-07/day-10-05_actual_candidates_and_answer_selection-01.png)

A symbolic token vocabulary and conditional EOS/0/1 support limit this lesson. Neither a plausible English prompt nor a low teacher-forced training loss establishes natural-language arithmetic.


### Prediction

## Exercise2 Does fitting remove the sampling problem

Before revealing the measured grid, predict what can remain wrong after the model learns six training sources. A new operand position, unseen instruction ID and new task family are separately frozen slices. Does repeating a strongly favored wrong answer create eight opportunities to correct it?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
initial,fitted,fresh_fit=fit_policy(items,10051)
assert fresh_fit==measured["runs"][0]["fit"]
saved_policy=measured["runs"][0]["policies"][1]
assert state_hash(fitted)==saved_policy["state_sha256"]
fresh_item=items[6]
fresh=[generate_candidate(fitted,fresh_item,measured["contract"],seed=10051,
       policy_id=saved_policy["checkpoint_id"],ordinal=i) for i in range(4)]
saved=next(p for p in saved_policy["pools"] if p["item_id"]==fresh_item["id"])["raw_candidates"][:4]
for a,b in zip(fresh,saved):
    assert a["token_ids"]==b["token_ids"]
    assert np.allclose(a["rescored_log_probabilities"],b["rescored_log_probabilities"],atol=1e-12,rtol=1e-12)
print("Fresh full fit and four coordinate candidates match saved non-timing evidence.")
fig,(loss_ax,grid_ax)=plt.subplots(1,2,figsize=(11,5.2),gridspec_kw={"width_ratios":[1,1.4]})
for run in measured["runs"]:
    loss_ax.semilogy([r["update"] for r in run["fit"]["history"]],
                    [r["loss"] for r in run["fit"]["history"]],label=str(run["seed"]))
loss_ax.set(xlabel="train-only update",ylabel="conditional response cross-entropy",title="All fixed80-update fits")
loss_ax.legend()
from matplotlib.colors import ListedColormap
grid=np.array([[1 if r["complete_success"] else 0 if r["status"]=="INCORRECT" else -1
              for r in p["graded_candidates"]] for p in saved_policy["pools"]])
grid_ax.imshow(grid,vmin=-1,vmax=1,cmap=ListedColormap(["#d8dce0","#b97139","#3d8268"]),aspect="auto")
grid_ax.set(xticks=range(8),xticklabels=range(1,9),xlabel="sample ordinal",
            yticks=range(18),yticklabels=[p["item_id"] for p in saved_policy["pools"]],
            title="Seed10051 final: wrong / invalid / correct")
grid_ax.axhline(5.5,color="#555555");grid_ax.axhline(9.5,color="#555555");grid_ax.axhline(13.5,color="#555555")
fig.tight_layout();plt.show()


### Reference explanation

Reference: use all three fixed80-update histories. Green cells below mean correct, one-integer, naturally terminated outputs under the independent evaluator; gray means invalid and brown means supported wrong. The selector never sees these colors.

![Saved training losses and actual candidate outcomes](../figures/chapter-07/day-10-05_actual_candidates_and_answer_selection-02.png)

The alias-template rows share held-out source groups with the direct rows. They are related observations, not extra independent subjects. The sum-positive slice is imbalanced: constant1 scores3/4 on its references, although an actual model still must emit the answer and EOS. No oracle stopping token is supplied here.


### Prediction

## Exercise3 Availability is not selection

Reuse prefixes1,2,4,8 of each saved pool. Should any-correct availability decrease when a nested prefix grows? Should majority or likelihood selection also be monotonic?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fig,axes=plt.subplots(2,3,figsize=(11,6.6),sharex=True,sharey=True)
for col,run in enumerate(measured["runs"]):
    for row,policy in enumerate(run["policies"]):
        ax=axes[row,col]
        for method in ("first","majority","mean_logp"):
            ys=[summary(policy,"heldout-source","attempts",n,method)["selected_success"] for n in N_VALUES]
            ax.plot(N_VALUES,ys,"o-",color=colors[method],label=method)
        oracle=[summary(policy,"heldout-source","attempts",n,"first")["oracle_any_complete"] for n in N_VALUES]
        assert all(b>=a for a,b in zip(oracle,oracle[1:]))
        ax.plot(N_VALUES,oracle,"s--",color=colors["oracle"],label="oracle availability")
        ax.set(title=f'{run["seed"]} · {policy["label"]}',xticks=N_VALUES,ylim=(-.03,1.03))
        if row==1:ax.set_xlabel("same-pool attempt prefixN")
        if col==0:ax.set_ylabel("complete success /4 sources")
axes[0,0].legend(fontsize=8)
fig.tight_layout();plt.show()


### Reference explanation

Reference: any-correct is a set-containment event and is monotonic for these nested pools. A selector can switch to a wrong candidate as new votes arrive. Mean log probability includes EOS and is not a truth score. The plot uses only the four predeclared source-heldout items, not training examples.

![Saved same pool source heldout selectors](../figures/chapter-07/day-10-05_actual_candidates_and_answer_selection-03.png)

At seed10052 the final policy's training item train-11 has five wrong1 votes and three correct0 votes. Oracle availability1 and majority success0 coexist. This is a real generated pool, not a categorical draw inserted into the experiment. Do not select a new seed because this result is inconvenient.


### Prediction

## Exercise4 Equal attempts are not equal token work

A failed attempt that emits three repeated numerals costs more output actions than immediate EOS. Suppose a complete attempt exceeds the remaining token budget. May we use it anyway, skip it for a cheaper later answer, or stop and charge it?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,3.8),sharey=True)
for run in measured["runs"]:
    for policy,marker in zip(run["policies"],["o","s"]):
        label=f'{run["seed"]} {policy["label"]}'
        rows=[summary(policy,"all","generation_tokens",b,"majority") for b in TOKEN_BUDGETS]
        accuracy=[r["selected_success"] for r in rows]
        axes[0].plot(TOKEN_BUDGETS,accuracy,marker+"-",label=label)
        axes[1].plot([r["generation_tokens"]/r["items"] for r in rows],accuracy,marker+"-")
        axes[2].plot([1000*r["wall_seconds"]/r["items"] for r in rows],accuracy,marker+"-")
axes[0].set(xlabel="declared output-token cap/item",ylabel="complete majority success /18items",
            title="Common token cap")
axes[1].set(xlabel="charged generation actions/item",title="Actual work includes overshoot")
axes[2].set(xlabel="observed CPU host ms/item",title="Generation +rescore +replay selector")
axes[0].legend(fontsize=7,ncol=2)
fig.tight_layout();plt.show()
pool=measured["runs"][0]["policies"][0]["pools"][0]["raw_candidates"]
accepted,attempted,rejections=budget_prefix(pool,tokens=4)
print("Accepted IDs:",[r["sample_id"] for r in accepted])
print("Charged IDs:",[r["sample_id"] for r in attempted],"boundary failures:",rejections)


### Reference explanation

Reference: this protocol charges and rejects the first whole-attempt overshoot and stops there. It is retrospective prefix replay, not token-level interruptible decoding. All selector arms pay the protocol's mandatory path rescoring; this is not an optimized first-answer serving baseline.

![Saved accuracy token and latency tradeoffs](../figures/chapter-07/day-10-05_actual_candidates_and_answer_selection-04.png)

These are sums of observed uncached CPU attempt times plus the replay selector, not new timed adaptive invocations. Prefill, subsequent full-prefix decoding and rescoring are recorded separately. The full campaign also includes fitting, parsing, every selector and file persistence. Generated actions are not FLOPs; latency should not be extrapolated to Spark inference frameworks.


### Prediction

## Exercise5 What kind of diversity matters

Eight independently seeded draws may still repeat one wrong answer. Conversely, two identical strings can result from independent draws; deleting duplicates changes the voting estimator. What do correctness agreement, wrong-answer agreement and output diversity each tell us?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4.1))
for ax,policy in zip(axes,measured["runs"][0]["policies"]):
    matrix=np.array([[np.nan if v is None else v for v in r] for r in policy["error_dependence"]["all"]["correlation"]])
    cmap=plt.get_cmap("RdBu_r").copy();cmap.set_bad("#d8dce0")
    im=ax.imshow(matrix,vmin=-1,vmax=1,cmap=cmap)
    ax.set(xticks=range(8),yticks=range(8),xticklabels=range(1,9),yticklabels=range(1,9),
           xlabel="candidate ordinal",ylabel="candidate ordinal",title=policy["label"]+" · error correlation")
fig.subplots_adjust(right=.83,wspace=.30)
cax=fig.add_axes([.87,.17,.023,.66])
fig.colorbar(im,cax=cax,label="descriptive fixed-item correlation")
plt.show()
for run in measured["runs"]:
    for policy in run["policies"]:
        fractions=[p["diagnostics"]["duplicate_raw_fraction"] for p in policy["pools"]]
        print(run["seed"],policy["label"],"mean within-pool raw duplicate fraction:",round(float(np.mean(fractions)),4))


### Reference explanation

Reference: the covariance below is across the18 fixed items at each sample ordinal. Source siblings remain related. A zero-variance ordinal has undefined correlation (gray), not correlation0. This is descriptive evidence about the frozen panel, not proof that samples satisfy or violate IID in a general population.

![Saved actual ordinal error dependence](../figures/chapter-07/day-10-05_actual_candidates_and_answer_selection-05.png)

Fitting can concentrate the output distribution. The measured final models have higher raw-string repetition, while source/template/family failures remain. Extra attempts can expose a correct alternative, reinforce a shared mistake, or produce no usable answer.


### Prediction

## Exercise6 Duplicate rules change the estimator

Use the actual seed10052 training failure. Compare counting every independently sampled vote, counting one vote per distinct answer, and using the likelihood ranker. What happens if gold fields are attached to the records?

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
failure=next(p for p in measured["runs"][1]["policies"][1]["pools"] if p["item_id"]=="train-11")
pool=failure["raw_candidates"]
for method in SELECTORS:
    decision=select(pool,method)
    injected=[dict(r,reference="0",correct=r["response_text"]=="0",problem={"gold":"0"}) for r in pool]
    assert decision==select(injected,method)
    print(method,decision)
print("Actual ordinal texts:",[r["response_text"] for r in pool])
assert any(r["complete_success"] for r in failure["graded_candidates"])
print("Oracle availability is true; it is never passed into select().")
# Controlled change: replace only this local prefix length; do not edit stored evidence.
for n in (1,2,4,8):
    print(n,select(pool[:n],"majority")["selected_sample_id"])


### Reference explanation

Reference: strict views must discard the fields and retain the same decision. The unique-support control weights0 and1 equally when both occur, reducing to earliest support in a tie; it is not a generally better self-consistency estimator. The longest-output selector is deliberately degenerate here because every eligible completion has one numeral and EOS.

## Evidence boundary

The three predeclared seeds produce864 retained actual candidates and1736 generated actions, including EOS, caps and invalids. They do not establish English reasoning, pretrained capability, reward-model calibration, optimized serving latency or learner mastery. The notebook recomputes a full fixed fit and four candidates, then uses the immutable historical full panel for its plots. A new recipe needs a new specification and run directory; do not quietly tune these held-out outcomes.
